# 简介

这是 **scikit-rf**（也称 `skrf`）的简要介绍。目标读者是那些已经安装了可用的 python 环境，并对 python 有一定了解的人。如果你是完全的 python 新手，请参阅 scipy 的 [入门指南](http://www.scipy.org/Getting_Started)。首先，导入 scikit-rf 模块 `skrf`，简写为 `rf`

In [ ]:
import skrf as rf

如果这产生了错误，请参阅[安装](Installation.ipynb)教程。

## 网络

`skrf` 中的核心对象是一个 N 端口微波 [Network][Network] 对象。[Network][Network] 可以通过多种方式创建，其中一种方式是从 touchstone 文件中存储的数据创建。

[Network]: ../api/network.rst

In [ ]:
ring_slot = rf.Network('data/ring slot.s2p')

如果你找不到 `ring slot.s2p`，只需从 `skrf.data` 模块导入它。

In [ ]:
from skrf.data import ring_slot  # noqa: F811

如果在命令行中输入网络对象，会打印出简短的描述

In [ ]:
ring_slot

微波 [Network](../api/network.rst) 的基本属性由以下属性提供：


* `Network.s` : 散射参数矩阵。
* `Network.z0`  : 端口阻抗矩阵。
* `Network.frequency`  : 频率对象。

[Network](../api/network.rst) 对象还有许多其他属性和方法，可以在其文档字符串中找到。如果你使用 IPython/Jupyter，那么这些属性和方法可以在命令行上通过"Tab"键补全。


	In [1]: ring_slot.s<TAB>
	ring_slot.s              ring_slot.s_arcl         ring_slot.s_im
	ring_slot.s11            ring_slot.s_arcl_unwrap  ring_slot.s_mag
	...

构建 Network 的其他方法详见 [Networks 教程](Networks.ipynb)。

### 线性运算

通过重载运算符可以访问对 s 参数的逐元素数学运算。为了说明，我们从 `skrf.data` 模块加载几个 `Networks`。

In [ ]:
short = rf.data.wr2p2_short
delayshort = rf.data.wr2p2_delayshort

它们 s 参数的复数差异通过以下方式计算

In [ ]:
short - delayshort

这返回一个新的 [Network](../api/network.rst)。其他算术运算符也被重载，

In [ ]:
short/delayshort

### 级联和去嵌入

2 端口网络的级联和去嵌入也可以通过运算符完成。级联通过幂运算符 `**` 完成。

In [ ]:
short = rf.data.wr2p2_short
line = rf.data.wr2p2_line

delayshort = line ** short

去嵌入可以通过级联网络的*逆*来完成。网络的逆通过属性 `Network.inv` 访问。

In [ ]:
short = line.inv ** delayshort

有关 [Network](../api/network.rst) 对象提供的功能的更多信息，例如插值、拼接、n 端口连接和 IO 支持，请参阅 [Networks](Networks.ipynb) 教程。

### 查找网络参数的最小值（或最大值）
通常，我们希望获取网络参数（s 参数、z 参数等）的最小值（或最大值）以及出现该值的频率。在 `scikit-rf` 中，网络参数以 [Numpy](https://numpy.org/) 数组的形式存储，形状为 ($N_F$, $N_p$, $N_p$)，其中 $N_F$ 是频率点的数量，$N_p$ 是网络的端口数：

In [ ]:
print(type(line.s))  # s-parameters are stored as a Numpy array

In [ ]:
print(line.s.shape)  # line is a 2-port Network defined on 201 frequency points

频率点定义在网络的 `frequency` 参数中：

In [ ]:
print(line.frequency)  # returns a Frequency object

频率值由 `frequency.f` 参数给出，或简单地使用 `.f`：

In [ ]:
line.f[0:10]  # the 10 first frequency points. Same than line.frequency.f[0:10]

作为 Numpy 数组，使用 `min()`（或 `max()`）方法可以找到 $S_{21}$ 参数幅度的最小值（或最大值）：

In [ ]:
import numpy as np

rs = rf.data.ring_slot  # another 2-port example

print(rs.s_mag[:,1,0].min())  # or .max() for maximum. Watch out that Python indexing starts at 0!

使用 Numpy 函数 [`argmin`](https://numpy.org/doc/stable/reference/generated/numpy.argmin.html?highlight=argmin#numpy.argmin) 可以找到 $S_{11}$ 参数幅度最小的频率：

In [ ]:
f_match = rs.f[np.argmin(rs.s_mag[:,0,0])]  # frequency for min(|S11|)
print(f_match)

## 绘图

**skrf** 有一个函数可以更新你的 [matplotlib rcParams](http://matplotlib.org/users/customizing.html)，使绘图看起来像本教程中显示的样式。

In [ ]:
# display plots in notebook
%matplotlib inline
import matplotlib.pyplot as plt

rf.stylely()

[Network](../api/network.rst) 类的方法提供了绘制网络参数各分量的便捷方式：

* `Network.plot_s_db()` : 以对数刻度绘制 s 参数的幅值
* `Network.plot_s_deg()` : 以度为单位绘制 s 参数的相位
* `Network.plot_s_smith()` : 在史密斯圆图上绘制复数 s 参数

要绘制 ``ring_slot`` 的所有四个 s 参数的幅值、相位和史密斯圆图。

In [ ]:
ring_slot.plot_s_db()

或绘制 $S_{12}$ 的相位

In [ ]:
ring_slot.plot_s_deg(m=0,n=1)

In [ ]:
ring_slot.plot_s_smith(lw=2)
plt.title('Big ole Smith Chart')

有关绘图的详细信息，请参阅 [Plotting](Plotting.ipynb) 教程

## 网络集

[NetworkSet](../api/networkSet.rst) 对象
表示一个无序的网络集合，并提供计算统计量和显示不确定度边界的方法。

可以从 [Networks](../api/network.rst) 的列表或字典创建 [NetworkSet](../api/networkSet.rst)。这可以通过 `rf.io.read_all()` 快速完成，它会加载目录中所有 skrf 可读的对象。参数 ``contains`` 用于仅加载匹配给定子字符串的文件。

In [ ]:
rf.io.read_all('data/', contains='ro')

这个字典可以直接传递给 [NetworkSet](../api/networkSet.rst) 构造函数，

In [ ]:
from skrf.networkSet import NetworkSet

ro_dict = rf.io.read_all('data/', contains='ro')
ro_ns = NetworkSet(ro_dict, name='ro set') # name is optional
ro_ns

[NetworkSet](../api/networkSet.rst) 类似于列表。

### 统计属性

可以通过访问 [NetworkSet](../api/networkSet.rst) 的属性来计算统计量。例如，要计算集合的复数平均值，访问 ``mean_s`` 属性

In [ ]:
ro_ns.mean_s

返回的结果存储在 [Network](../api/network.rst) 的 s 参数中，无论输出类型如何。类似地，要计算集合的复数标准差，

In [ ]:
ro_ns.std_s

因为这些方法返回一个 [Network](../api/network.rst) 对象，所以结果可以以与 [Network](../api/network.rst) 相同的方式保存或绘制。要绘制集合标准差的幅值，

In [ ]:
ro_ns.std_s.plot_s_mag(label='S11')
plt.ylabel('Standard Deviation')
plt.title('Standard Deviation of RO')

### 绘制不确定度边界

任何网络参数的不确定度边界都可以通过以下方法绘制

In [ ]:
ro_ns.plot_uncertainty_bounds_s_db(label='S11');

有关更多信息，请参阅 [networkset](NetworkSet.ipynb) 教程。

## 虚拟仪器

[skrf.vi](../api/vi/index.rst) 模块提供了与某些仪器通信的类。目前，这只支持 VNA，但将来可能会添加更多支持。有关更多信息，请参阅 [虚拟仪器](VirtualInstruments.ipynb) 教程。

使用 `PNA` 类检索一些 s 参数数据并绘图的示例

    from skrf.vi.vna.keysight import PNA
    instr = PNA(address="TCPIP0::10.0.0.1::INSTR") 

    freq = Frequency(start=2.3e9, stop=2.6e9, npoints=401, unit='hz')
    instr.frequency = freq

    ntwk = instr.get_snp_network(ports=(1,2))
    ntwk.s21.plot_s_db()

## 校准

校准通过 [Calibration](../api/calibration/index.rst) 类执行。在大多数情况下，创建 [Calibration](../api/calibration/index.rst) 对象至少需要两个信息：

*   测量 [Network](../api/network.rst) 的列表
*   理想 [Network](../api/network.rst) 的列表

每个列表中的 [Network](../api/network.rst) 元素必须都相似（相同的端口数、频率信息等），并且必须相互对齐，即理想列表的第一个元素必须对应测量列表的第一个元素。

下面是一个说明如何创建 [Calibration](../api/calibration/index.rst) 的示例脚本。

### 单端口校准

    import skrf as rf
    from skrf.calibration import OnePort
    
    my_ideals = rf.io.read_all('ideals/')
    my_measured = rf.io.read_all('measured/')
    duts = rf.io.read_all('measured/')
    
    ## 创建 Calibration 实例
    cal = OnePort(
        ideals = [my_ideals[k] for k in ['short','open','load']],
        measured = [my_measured[k] for k in ['short','open','load']],
        )
    
    caled_duts = [cal.apply_cal(dut) for dut in duts.values()]
    
更多详细信息和示例，请参阅 [Calibration](Calibration.ipynb) 教程。

## 传输线介质

简单传输线网络可以通过 [Media](../api/media/index.rst) 类的方法创建，该类表示给定介质的传输线对象。构造完成后，[Media](../api/media/index.rst) 对象包含生成微波电路所需的必要属性，如 ``传播常数`` 和 ``特性阻抗``。

基本用法如下：

### CPW（共面波导）

In [ ]:
from skrf import Frequency
from skrf.media import CPW, Coaxial

freq = Frequency(75, 110, 101, 'GHz')
cpw =  CPW(freq, w = 10e-6, s = 5e-6, ep_r = 10.6)
cpw

In [ ]:
cpw.line(d=90,unit='deg', name='line')

### Coax（同轴）

In [ ]:
freq = Frequency(1, 10, 101, 'GHz')
coax = Coaxial(frequency = freq, Dint = 1e-3, Dout = 2e-3)
coax